# FedLEASE-FinBERT: Federated Financial Sentiment Analysis

**Method:** Two-phase clustered LoRA-MoE with adaptive top-M routing  
**Datasets:** Financial PhraseBank + Twitter Financial News Sentiment  
**Model:** ProsusAI/FinBERT — frozen backbone, ~0.27% trainable params

---

## How to run

| Step | Action |
|---|---|
| 0 | GPU check |
| 1 | Install deps (pins datasets<3.0) |
| 2 | Upload fedlease.zip |
| 3 | Patch dataset_loader.py (run once) |
| 4 | Set HF token (optional but faster) |
| 5 | Configure preset |
| 6 | Run pipeline |
| 7 | View results |
| 8 | Download outputs |

**Why Step 3?** `datasets>=3.0` removed loading-script support.  
The fix rewrites the loader to download the raw PhraseBank zip directly  
via `huggingface_hub` (no `load_dataset()` call — no script ban applies).

---
## Step 0 — GPU Check

In [ ]:
import subprocess, sys, torch

try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except FileNotFoundError:
    print('No GPU found. Runtime -> Change runtime type -> T4 GPU')

print(f'Python {sys.version}')
print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  '
          f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

try:
    import google.colab
    print('Running in Google Colab')
except ImportError:
    print('Not in Colab')


---
## Step 1 — Install Dependencies

**Important:** We pin `datasets<3.0` because datasets 3.x removed
loading-script support (breaks `financial_phrasebank`).  
Run once per session.

In [ ]:
%%capture install_out

# Pin datasets to 2.x — datasets 3.x removed loading-script support
# (The fix in Step 3 also bypasses load_dataset entirely for PhraseBank,
#  but pinning ensures everything is consistent)
!pip install -q "datasets>=2.15.0,<3.0.0"

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers>=4.36.0 peft>=0.7.1 accelerate>=0.25.0 tokenizers>=0.15.0
!pip install -q scikit-learn>=1.3.2 scipy>=1.11.4 numpy>=1.26.0 pandas>=2.1.0
!pip install -q matplotlib>=3.8.0 seaborn>=0.13.0
!pip install -q omegaconf>=2.3.0 PyYAML>=6.0.1 tqdm>=4.66.0 einops>=0.7.0
!pip install -q requests>=2.31.0 pyarrow>=14.0.0 huggingface_hub>=0.20.0

print("Installation done. Verifying...")


In [ ]:
import importlib, torch

pkgs = ['torch','transformers','datasets','sklearn','scipy',
        'numpy','matplotlib','omegaconf','tqdm','requests','pyarrow']
for pkg in pkgs:
    try:
        m = importlib.import_module(pkg)
        print(f'  OK {pkg:20s} {getattr(m,"__version__","ok")}')
    except ImportError:
        print(f'  MISSING {pkg} -- re-run Step 1')

import datasets
dv = tuple(int(x) for x in datasets.__version__.split('.')[:2])
if dv >= (3, 0):
    print(f'\nWARNING: datasets {datasets.__version__} is >=3.0')
    print('Run: !pip install -q "datasets>=2.15.0,<3.0.0" then restart runtime')
else:
    print(f'\nOK datasets {datasets.__version__} is <3.0 -- loading scripts supported')

print(f'CUDA: {torch.cuda.is_available()}', end='  ')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print()


---
## Step 2 — Upload FedLEASE Code

On your Mac, create the zip:
```bash
cd ~/Desktop/research_paper
zip -r fedlease.zip fedlease/
```
Then run this cell and click **Choose Files**.

In [ ]:
import os, sys

FEDLEASE_ROOT = '/content/fedlease'

if os.path.exists(FEDLEASE_ROOT):
    print(f'fedlease/ already present at {FEDLEASE_ROOT}')
else:
    try:
        from google.colab import files
        print('Select your fedlease.zip:')
        uploaded = files.upload()
        zname = list(uploaded.keys())[0]
        # Try standard unzip first
        ret = os.system(f'unzip -q "{zname}" -d /content/')
        if not os.path.exists(FEDLEASE_ROOT):
            # zip may have been made from inside fedlease/ dir
            os.makedirs(FEDLEASE_ROOT, exist_ok=True)
            os.system(f'unzip -q "{zname}" -d {FEDLEASE_ROOT}')
    except ImportError:
        print('Not in Colab. Place fedlease/ at /content/fedlease manually.')

if FEDLEASE_ROOT not in sys.path:
    sys.path.insert(0, FEDLEASE_ROOT)

# Verify key files
required = [
    'data/dataset_loader.py', 'data/data_partitioner.py',
    'models/finbert_lora_moe.py', 'models/adaptive_router.py',
    'training/fedlease_pipeline.py', 'federated/client.py',
    'federated/server.py', 'clustering/clustering.py',
    'configs/finbert_fedlease.yaml',
]
all_ok = True
for f in required:
    ok = os.path.exists(os.path.join(FEDLEASE_ROOT, f))
    print(f'  {"OK" if ok else "MISSING"} {f}')
    if not ok: all_ok = False
print(f'\n{"All files found" if all_ok else "Some files missing -- check zip contents"}')


---
## Step 3 — Patch dataset_loader.py

Run once after every upload. Rewrites the loader to download the raw
PhraseBank zip via `huggingface_hub.hf_hub_download()` — completely
bypasses `load_dataset()` so the loading-script ban never applies.

In [ ]:
import os, sys, py_compile

FEDLEASE_ROOT = '/content/fedlease'
loader_path   = os.path.join(FEDLEASE_ROOT, 'data', 'dataset_loader.py')

FIXED_LOADER = '"""Dataset loader for FedLEASE.\n\nFinancial PhraseBank loading notes\n------------------------------------\nThe ``financial_phrasebank`` HuggingFace repo uses a legacy loading script\nthat ``datasets>=3.0`` refuses to execute.  Neither ``trust_remote_code=True``\n(now rejected) nor alternative repo mirrors solve the problem because every\nknown mirror also ships the ``.py`` script file.\n\nSolution: bypass ``load_dataset()`` entirely for PhraseBank.\nWe use ``huggingface_hub.hf_hub_download`` to download the raw zip that the\nloading script itself would have downloaded, then parse it directly.\n``huggingface_hub`` is always available (it is a dependency of\n``transformers``) and never executes scripts — it just fetches files.\n\nTwitter Financial News Sentiment does NOT use a loading script, so\n``load_dataset()`` works fine for that dataset.\n"""\n\nfrom __future__ import annotations\n\nimport io\nimport os\nimport zipfile\nfrom typing import Dict, List, Optional, Tuple\n\nimport numpy as np\nfrom datasets import load_dataset, Dataset, DatasetDict\nfrom sklearn.model_selection import train_test_split\n\nLABEL_NAMES            = ["negative", "neutral", "positive"]\nLABEL_MAP_PHRASEBANK   = {"negative": 0, "neutral": 1, "positive": 2}\nLABEL_MAP_TWITTER      = {"Bearish": 0, "Neutral": 1, "Bullish": 2}\nPHRASEBANK_INT_TO_NAME = {0: "negative", 1: "neutral", 2: "positive"}\n\n\n# ---------------------------------------------------------------------------\n# Internal helpers\n# ---------------------------------------------------------------------------\n\ndef _hf_token() -> Optional[str]:\n    """Return HuggingFace token from env if set."""\n    return (\n        os.environ.get("HF_TOKEN")\n        or os.environ.get("HUGGING_FACE_HUB_TOKEN")\n        or os.environ.get("HUGGINGFACE_TOKEN")\n    )\n\n\ndef _parse_phrasebank_txt(content: str) -> Tuple[List[str], List[int]]:\n    """Parse Financial PhraseBank raw text format.\n\n    Each line is ``sentence@sentiment_label`` where label is one of\n    ``negative``, ``neutral``, ``positive``.\n    """\n    lmap = {"negative": 0, "neutral": 1, "positive": 2}\n    texts: List[str]  = []\n    labels: List[int] = []\n    for line in content.splitlines():\n        line = line.strip()\n        if not line or "@" not in line:\n            continue\n        *parts, label_str = line.rsplit("@", 1)\n        sentence  = "@".join(parts).strip()   # rejoin in case sentence had @\n        label_str = label_str.strip().lower()\n        if sentence and label_str in lmap:\n            texts.append(sentence)\n            labels.append(lmap[label_str])\n    return texts, labels\n\n\ndef _load_phrasebank_via_hf_hub(config: str = "sentences_allagree") -> Tuple[List[str], List[int]]:\n    """Download Financial PhraseBank directly via huggingface_hub.\n\n    Approach:\n    1. List all files in the ``financial_phrasebank`` dataset repo.\n    2. Download whichever zip / txt file matches the requested config.\n    3. Parse the ``sentence@label`` text format.\n\n    This never calls ``load_dataset()`` so the loading-script ban is irrelevant.\n    """\n    from huggingface_hub import hf_hub_download, list_repo_files  # always available\n\n    token = _hf_token()\n    repo_id = "financial_phrasebank"\n\n    print(f"  [loader] Listing files in {repo_id}...", flush=True)\n    try:\n        all_files = list(list_repo_files(repo_id, repo_type="dataset", token=token))\n    except Exception as exc:\n        raise RuntimeError(\n            f"Cannot list files in {repo_id}: {exc}\\n"\n            "Check your internet connection or set HF_TOKEN env var."\n        ) from exc\n\n    print(f"  [loader] Repo files: {all_files}", flush=True)\n\n    # ── Try 1: look for a pre-extracted .txt matching the config ────────────\n    config_key = config.replace("sentences_", "")   # e.g. "allagree"\n    txt_candidates = [\n        f for f in all_files\n        if config_key in f.lower() and f.endswith(".txt")\n    ]\n    if txt_candidates:\n        local_path = hf_hub_download(\n            repo_id=repo_id,\n            filename=txt_candidates[0],\n            repo_type="dataset",\n            token=token,\n        )\n        with open(local_path, "r", encoding="latin-1") as fh:\n            content = fh.read()\n        texts, labels = _parse_phrasebank_txt(content)\n        if texts:\n            print(f"  [loader] Loaded {len(texts)} sentences from {txt_candidates[0]}", flush=True)\n            return texts, labels\n\n    # ── Try 2: download zip and extract matching txt ─────────────────────────\n    zip_candidates = [f for f in all_files if f.endswith(".zip")]\n    if not zip_candidates:\n        raise RuntimeError(\n            f"No .zip or matching .txt found in {repo_id}.\\n"\n            f"Files present: {all_files}"\n        )\n\n    zip_name   = zip_candidates[0]\n    local_path = hf_hub_download(\n        repo_id=repo_id,\n        filename=zip_name,\n        repo_type="dataset",\n        token=token,\n    )\n    print(f"  [loader] Downloaded {zip_name}", flush=True)\n\n    with zipfile.ZipFile(local_path) as zf:\n        zip_entries = zf.namelist()\n        print(f"  [loader] Zip contents: {zip_entries}", flush=True)\n\n        # Find the txt file matching the requested config\n        txt_entry = next(\n            (\n                n for n in zip_entries\n                if config_key in n.lower() and n.endswith(".txt")\n            ),\n            None,\n        )\n        if txt_entry is None:\n            raise RuntimeError(\n                f"Could not find a .txt for config={config!r} inside {zip_name}.\\n"\n                f"Zip contents: {zip_entries}"\n            )\n\n        with zf.open(txt_entry) as fh:\n            content = fh.read().decode("latin-1")\n\n    texts, labels = _parse_phrasebank_txt(content)\n    print(f"  [loader] Parsed {len(texts)} sentences from {txt_entry}", flush=True)\n    return texts, labels\n\n\n# ---------------------------------------------------------------------------\n# Public class\n# ---------------------------------------------------------------------------\n\nclass FinancialDatasetLoader:\n    """Loads and stratified-splits both financial-NLP datasets."""\n\n    def __init__(\n        self,\n        val_ratio:  float           = 0.10,\n        test_ratio: float           = 0.20,\n        seed:       int             = 42,\n        cache_dir:  Optional[str]   = None,\n    ) -> None:\n        self.val_ratio  = val_ratio\n        self.test_ratio = test_ratio\n        self.seed       = seed\n        self.cache_dir  = cache_dir\n\n    # ------------------------------------------------------------------\n    def load_phrasebank(self) -> DatasetDict:\n        """Return {train, val, test} for Financial PhraseBank (sentences_allagree).\n\n        Downloads the raw zip directly via ``huggingface_hub`` — bypasses\n        ``load_dataset()`` and therefore the loading-script ban in datasets>=3.0.\n        """\n        texts, labels = _load_phrasebank_via_hf_hub("sentences_allagree")\n        return self._split(texts, labels, source_tag="phrasebank")\n\n    def load_twitter(self) -> DatasetDict:\n        """Return {train, val, test} for Twitter Financial News Sentiment."""\n        raw = load_dataset(\n            "zeroshot/twitter-financial-news-sentiment",\n            cache_dir=self.cache_dir,\n        )\n        # Native:  0=Bearish, 1=Bullish, 2=Neutral\n        # Unified: 0=negative, 1=neutral, 2=positive\n        label_remap    = {0: 0, 1: 2, 2: 1}\n        all_texts:  List[str]  = []\n        all_labels: List[int]  = []\n        for split in ["train", "validation"]:\n            if split not in raw:\n                continue\n            all_texts.extend(list(raw[split]["text"]))\n            all_labels.extend([label_remap[int(lbl)] for lbl in raw[split]["label"]])\n        return self._split(all_texts, all_labels, source_tag="twitter")\n\n    def load_all(self) -> Tuple[DatasetDict, DatasetDict]:\n        return self.load_phrasebank(), self.load_twitter()\n\n    def get_statistics(self, dataset_splits: DatasetDict) -> Dict:\n        stats: Dict = {}\n        for split_name, split_data in dataset_splits.items():\n            labels = split_data["label"]\n            unique, counts = np.unique(labels, return_counts=True)\n            stats[split_name] = {\n                "total": len(labels),\n                "class_counts": {\n                    LABEL_NAMES[int(u)]: int(c) for u, c in zip(unique, counts)\n                },\n                "class_ratios": {\n                    LABEL_NAMES[int(u)]: float(c) / len(labels)\n                    for u, c in zip(unique, counts)\n                },\n            }\n        return stats\n\n    # ------------------------------------------------------------------\n    def _split(\n        self,\n        texts:      List[str],\n        labels:     List[int],\n        source_tag: str,\n    ) -> DatasetDict:\n        indices = list(range(len(texts)))\n\n        train_val_idx, test_idx = train_test_split(\n            indices,\n            test_size=self.test_ratio,\n            stratify=labels,\n            random_state=self.seed,\n        )\n        train_val_labels = [labels[i] for i in train_val_idx]\n        val_frac = self.val_ratio / (1.0 - self.test_ratio)\n\n        train_idx, val_idx = train_test_split(\n            train_val_idx,\n            test_size=val_frac,\n            stratify=train_val_labels,\n            random_state=self.seed,\n        )\n\n        def make_split(idxs: List[int]) -> Dataset:\n            return Dataset.from_dict({\n                "text":   [texts[i]  for i in idxs],\n                "label":  [labels[i] for i in idxs],\n                "source": [source_tag] * len(idxs),\n            })\n\n        return DatasetDict({\n            "train": make_split(train_idx),\n            "val":   make_split(val_idx),\n            "test":  make_split(test_idx),\n        })\n'

os.makedirs(os.path.dirname(loader_path), exist_ok=True)
with open(loader_path, 'w') as f:
    f.write(FIXED_LOADER)

try:
    py_compile.compile(loader_path, doraise=True)
    print('OK  dataset_loader.py patched')
    print('    PhraseBank: uses hf_hub_download + zip parse')
    print('    Twitter   : uses load_dataset (no script, always works)')
except py_compile.PyCompileError as e:
    print(f'SYNTAX ERROR: {e}'); raise

# Reload module so the fix is live immediately
for k in list(sys.modules.keys()):
    if 'dataset_loader' in k: del sys.modules[k]
if FEDLEASE_ROOT not in sys.path: sys.path.insert(0, FEDLEASE_ROOT)
from data.dataset_loader import FinancialDatasetLoader, _load_phrasebank_via_hf_hub
print('OK  FinancialDatasetLoader imported -- ready')


---
## Step 4 — HuggingFace Token (optional but recommended)

Setting your HF token raises rate limits and speeds up downloads.  
Get yours at https://huggingface.co/settings/tokens

In [ ]:
import os

HF_TOKEN = ""  # paste your token here, e.g. "hf_xxxxxxxxxxxx"

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Logged in to HuggingFace Hub")
    except Exception as e:
        print(f"HF login warning (non-fatal): {e}")
else:
    print("No token set -- using anonymous access (may be slower)")


---
## Step 5 — Configure Experiment

| Preset | Rounds | GPU time |
|---|---|---|
| `debug`  | 3  | ~5 min |
| `quick`  | 10 | ~25 min |
| `medium` | 15 | ~40 min |
| `full`   | 25 | ~60 min |

In [ ]:
import torch, os, yaml

PRESET = 'quick'   # change to 'debug' / 'medium' / 'full'

PRESETS = {
    'debug':  dict(num_rounds=3,  local_epochs=1, warmup_epochs=1, batch_size=16, max_experts=4),
    'quick':  dict(num_rounds=10, local_epochs=1, warmup_epochs=2, batch_size=32, max_experts=6),
    'medium': dict(num_rounds=15, local_epochs=2, warmup_epochs=3, batch_size=32, max_experts=8),
    'full':   dict(num_rounds=25, local_epochs=2, warmup_epochs=3, batch_size=32, max_experts=8),
}
assert PRESET in PRESETS
P = PRESETS[PRESET]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Preset  : {PRESET}')
print(f'Rounds  : {P["num_rounds"]}')
print(f'Warmup  : {P["warmup_epochs"]} epochs/client')
print(f'Batch   : {P["batch_size"]}')
print(f'Device  : {DEVICE}')
if DEVICE == 'cuda': print(f'GPU     : {torch.cuda.get_device_name(0)}')


In [ ]:
import yaml, os

OUTPUT_DIR = '/content/fedlease_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

cfg_dict = {
    'model':     {'name': 'ProsusAI/finbert', 'num_labels': 3, 'max_length': 128},
    'lora':      {'rank': 4, 'alpha': 8, 'dropout': 0.1,
                  'target_modules': ['query', 'value']},
    'federated': {
        'num_clients': 10,
        'num_rounds':    P['num_rounds'],
        'local_epochs':  P['local_epochs'],
        'batch_size':    P['batch_size'],
        'warmup_epochs': P['warmup_epochs'],
        'max_experts':   P['max_experts'],
        'min_experts': 2,
    },
    'training': {
        'learning_rate': 3.0e-4, 'weight_decay': 0.01,
        'warmup_ratio': 0.1, 'max_grad_norm': 1.0,
        'mixed_precision': DEVICE == 'cuda', 'seed': 42,
    },
    'clustering': {'min_clusters': 2, 'max_clusters': P['max_experts'], 'linkage': 'average'},
    'data': {
        'dataset_phrasebank': 'financial_phrasebank',
        'dataset_phrasebank_config': 'sentences_allagree',
        'dataset_twitter': 'zeroshot/twitter-financial-news-sentiment',
        'val_ratio': 0.10, 'test_ratio': 0.20,
        'partition_strategy': 'heterogeneous',
        'num_phrasebank_clients': 5, 'num_twitter_clients': 5,
        'dirichlet_alpha': 0.5,
    },
    'paths': {
        'output_dir':     OUTPUT_DIR,
        'checkpoint_dir': os.path.join(OUTPUT_DIR, 'checkpoints'),
        'log_dir':        os.path.join(OUTPUT_DIR, 'logs'),
        'plot_dir':       os.path.join(OUTPUT_DIR, 'plots'),
        'result_dir':     os.path.join(OUTPUT_DIR, 'results'),
    },
    'logging': {'level': 'INFO', 'use_wandb': False,
                'project_name': 'fedlease-finbert',
                'experiment_name': f'colab_{PRESET}'},
}

CONFIG_PATH = '/content/fedlease_config.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg_dict, f, default_flow_style=False)
for p in cfg_dict['paths'].values():
    os.makedirs(p, exist_ok=True)
print(f'Config  -> {CONFIG_PATH}')
print(f'Outputs -> {OUTPUT_DIR}')


---
## Step 6 — Run FedLEASE Pipeline

**Phase A — Warmup:** each client trains a local LoRA on private data.  
**Clustering:** server clusters B-matrices by cosine similarity → silhouette-optimal M*.  
**Phase B — Federated:** T rounds, adaptive top-M router, cluster-wise FedAvg.

In [ ]:
import sys, os, torch, numpy as np

FEDLEASE_ROOT = '/content/fedlease'
if FEDLEASE_ROOT not in sys.path:
    sys.path.insert(0, FEDLEASE_ROOT)

# Clear stale cached modules
stale = [k for k in sys.modules if any(
    k.startswith(m) for m in
    ['data.','models.','training.','federated.','clustering.','utils.','visualization.']
)]
for k in stale:
    del sys.modules[k]

from data.dataset_loader        import FinancialDatasetLoader
from data.data_partitioner      import FederatedDataPartitioner
from data.preprocessing         import FinancialPreprocessor
from training.fedlease_pipeline import FedLEASEPipeline
from utils.config               import load_config
from utils.logging_utils        import setup_logger
from utils.seed                 import set_seed

print('All imports OK')
cfg    = load_config(CONFIG_PATH)
device = torch.device(DEVICE)
set_seed(cfg.training.seed)
logger = setup_logger('fedlease_colab', log_dir=cfg.paths.log_dir, level='INFO')
print(f'Config loaded  device={device}')


In [ ]:
print('\n' + '='*60)
print('LOADING & PARTITIONING DATA')
print('='*60)

loader = FinancialDatasetLoader(
    val_ratio=cfg.data.val_ratio,
    test_ratio=cfg.data.test_ratio,
    seed=cfg.training.seed,
)

print('Loading Financial PhraseBank (hf_hub_download + zip parse)...', flush=True)
phraseb_splits = loader.load_phrasebank()

print('Loading Twitter Financial News Sentiment...', flush=True)
twitter_splits = loader.load_twitter()

pb = loader.get_statistics(phraseb_splits)
tw = loader.get_statistics(twitter_splits)
print(f'\nDataset Summary:')
print(f'  PhraseBank train={pb["train"]["total"]:,}  val={pb["val"]["total"]:,}  test={pb["test"]["total"]:,}')
print(f'  Twitter    train={tw["train"]["total"]:,}  val={tw["val"]["total"]:,}  test={tw["test"]["total"]:,}')

print('\nPartitioning into 10 clients (5 PhraseBank + 5 Twitter)...')
partitioner = FederatedDataPartitioner(seed=cfg.training.seed)
client_data_list = partitioner.partition(
    phrasebank_splits=phraseb_splits,
    twitter_splits=twitter_splits,
    num_clients=cfg.federated.num_clients,
    strategy=cfg.data.partition_strategy,
    num_phrasebank_clients=cfg.data.num_phrasebank_clients,
    num_twitter_clients=cfg.data.num_twitter_clients,
    dirichlet_alpha=cfg.data.dirichlet_alpha,
)

stats = partitioner.get_partition_stats(client_data_list)
print('\nClient Partitions:')
for cid, s in stats.items():
    print(f'  {cid} [{s["source"]:12s}] '
          f'train={s["train_size"]:4d}  val={s["val_size"]:3d}  '
          f'dist={s["label_distribution"]}')
print('\nData ready')


In [ ]:
preprocessor = FinancialPreprocessor(
    model_name=cfg.model.name, max_length=cfg.model.max_length)

pbt = phraseb_splits['test']
twt = twitter_splits['test']
test_loaders = {
    'phrasebank_test': preprocessor.make_dataloader(
        pbt['text'], pbt['label'], batch_size=cfg.federated.batch_size, shuffle=False),
    'twitter_test': preprocessor.make_dataloader(
        twt['text'], twt['label'], batch_size=cfg.federated.batch_size, shuffle=False),
}
print(f'PhraseBank test : {len(pbt["text"]):,} examples')
print(f'Twitter test    : {len(twt["text"]):,} examples')


In [ ]:
import time

print('\n' + '='*60)
print('FEDLEASE TRAINING')
print(f'  Warmup epochs   : {cfg.federated.warmup_epochs}')
print(f'  Federated rounds: {cfg.federated.num_rounds}')
print(f'  Local epochs    : {cfg.federated.local_epochs}')
print(f'  Max experts     : {cfg.federated.max_experts}')
print('='*60)

t0 = time.time()
pipeline = FedLEASEPipeline(
    config=cfg, client_data_list=client_data_list,
    test_loaders=test_loaders, device=device,
)
results  = pipeline.run()
elapsed  = time.time() - t0
comm     = results.get('comm_stats', {})

print(f'\nDone in {elapsed/60:.1f} min')
print('='*60)
print(f'  M*             : {pipeline.server.n_experts}')
print(f'  Rounds         : {comm.get("total_rounds","?")}')
print(f'  Total comm     : {comm.get("total_bytes_MB",0):.2f} MB')
for ds, m in results.get('final_metrics', {}).items():
    a = m.get('aggregate', {})
    print(f'  [{ds:20s}]  acc={a.get("mean_accuracy",0):.4f}  '
          f'macro-F1={a.get("mean_macro_f1",0):.4f}')


---
## Step 7 — Results & Plots

In [ ]:
import pandas as pd
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120

print('=' * 60)
print('FINAL METRICS')
print('=' * 60)

rows = []
for ds, m in results.get('final_metrics', {}).items():
    a = m.get('aggregate', {})
    rows.append({'Dataset': ds,
                 'Accuracy':    f"{a.get('mean_accuracy',    0):.4f}",
                 'Macro-F1':    f"{a.get('mean_macro_f1',    0):.4f}",
                 'Weighted-F1': f"{a.get('mean_weighted_f1', 0):.4f}"})
if rows:
    print(pd.DataFrame(rows).to_string(index=False))

print('\nPaper Targets:')
for ds, t in [('phrasebank_test', {'accuracy': 0.92, 'macro_f1': 0.88}),
              ('twitter_test',    {'accuracy': 0.82, 'macro_f1': 0.80})]:
    if ds in results.get('final_metrics', {}):
        a   = results['final_metrics'][ds].get('aggregate', {})
        acc = a.get('mean_accuracy', 0)
        f1  = a.get('mean_macro_f1', 0)
        print(f'  {ds}:')
        print(f'    {"OK" if acc >= t["accuracy"] else "!!"} Accuracy {acc:.4f} (>= {t["accuracy"]})')
        print(f'    {"OK" if f1  >= t["macro_f1"]  else "!!"} Macro-F1 {f1:.4f}  (>= {t["macro_f1"]})')


In [ ]:
import matplotlib.pyplot as plt, numpy as np

try:
    curves  = pipeline.get_training_curves()
    va = curves.get('avg_val_accuracy', [])
    vl = curves.get('avg_val_loss', [])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('FedLEASE Training Curves', fontsize=14, fontweight='bold')

    ax = axes[0]
    if va:
        ax.plot(range(1, len(va)+1), va, 'b-o', lw=2, ms=4, label='FedLEASE')
        ax.axhline(0.88, color='green',  ls='--', alpha=.7, label='PhraseBank target')
        ax.axhline(0.80, color='orange', ls='--', alpha=.7, label='Twitter target')
        ax.set_xlabel('Round'); ax.set_ylabel('Val Accuracy')
        ax.set_title('Validation Accuracy'); ax.set_ylim(0, 1)
        ax.legend(); ax.grid(True, alpha=.3)

    ax = axes[1]
    if vl:
        ax.plot(range(1, len(vl)+1), vl, 'r-o', lw=2, ms=4)
        ax.set_xlabel('Round'); ax.set_ylabel('Val Loss')
        ax.set_title('Validation Loss'); ax.grid(True, alpha=.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'plots', 'training_curves.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Training curve error: {e}')


In [ ]:
import matplotlib.pyplot as plt, numpy as np, os

dp = os.path.join(OUTPUT_DIR, 'distance_matrix.npy')
if os.path.exists(dp):
    dm = np.load(dp)
    cl = np.load(os.path.join(OUTPUT_DIR, 'cluster_labels.npy'))
    sd = np.load(os.path.join(OUTPUT_DIR, 'silhouette_scores.npy'))
    ss = {int(k): float(v) for k, v in sd}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('FedLEASE Clustering', fontsize=14, fontweight='bold')

    ax = axes[0]
    im = ax.imshow(dm, cmap='YlOrRd', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='cosine distance')
    n   = len(dm)
    src = [c['source'] for c in pipeline.client_data_list]
    lbl = [f"C{c['client_id']}\n({s[:2].upper()})"
           for c, s in zip(pipeline.client_data_list, src)]
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(lbl, fontsize=7); ax.set_yticklabels(lbl, fontsize=7)
    ax.set_title('B-Matrix Cosine Distance')
    COLS = ['royalblue','tomato','seagreen','mediumpurple','darkorange']
    for i, c in enumerate(cl):
        ax.get_xticklabels()[i].set_color(COLS[c % len(COLS)])
        ax.get_yticklabels()[i].set_color(COLS[c % len(COLS)])

    ax = axes[1]
    ks  = sorted(ss); svs = [ss[k] for k in ks]; bk = max(ss, key=ss.get)
    ax.plot(ks, svs, 'b-o', lw=2, ms=6)
    ax.axvline(bk, color='red', ls='--', alpha=.7, label=f'M*={bk} S={ss[bk]:.3f}')
    ax.set_xlabel('M'); ax.set_ylabel('Silhouette')
    ax.set_title('Silhouette vs M'); ax.set_xticks(ks)
    ax.legend(); ax.grid(True, alpha=.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'plots', 'clustering.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nCluster Assignment (M*={pipeline.server.n_experts}):')
    for c, x in zip(pipeline.client_data_list, cl):
        print(f'  C{c["client_id"]:2d} [{c["source"]:12s}] -> Cluster {x}')
else:
    print('distance_matrix.npy not found')


In [ ]:
import matplotlib.pyplot as plt, numpy as np

try:
    bpr = pipeline.server.comm_stats.get('bytes_per_round', [])
    cs  = pipeline.server.communication_summary()
    print('Communication Summary:')
    for k, v in cs.items(): print(f'  {k}: {v}')

    if bpr:
        mb  = np.array(bpr) / 1e6
        cum = np.cumsum(mb)
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        fig.suptitle('FedLEASE Communication', fontsize=14, fontweight='bold')

        ax = axes[0]
        ax.bar(range(1, len(mb)+1), mb, color='steelblue', alpha=.85)
        ax.axhline(mb.mean(), color='red', ls='--', label=f'Avg {mb.mean():.2f} MB')
        ax.set_xlabel('Round'); ax.set_ylabel('MB')
        ax.set_title('Per-Round Upload'); ax.legend(); ax.grid(True, alpha=.3, axis='y')

        ax = axes[1]
        ax.plot(range(1, len(cum)+1), cum, 'g-o', lw=2, ms=4)
        ax.axhline(200, color='red', ls='--', alpha=.7, label='200 MB paper target')
        ax.set_xlabel('Round'); ax.set_ylabel('MB')
        ax.set_title('Cumulative Upload'); ax.legend(); ax.grid(True, alpha=.3)

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'plots', 'communication.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
except Exception as e:
    print(f'Comm plot error: {e}')


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

try:
    accs, f1s = {}, {}
    for ds, m in results.get('final_metrics', {}).items():
        for cid, cm in m.get('per_client', {}).items():
            if cm and 'accuracy' in cm:
                key = f"{cid}\n({ds[:4]})"
                accs[key] = cm['accuracy']
                f1s[key]  = cm.get('macro_f1', 0)

    if accs:
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        fig.suptitle('Per-Client Performance', fontsize=13, fontweight='bold')
        for ax, vals, label in [(axes[0], accs, 'Accuracy'), (axes[1], f1s, 'Macro-F1')]:
            cols = ['#4472C4' if 'phra' in k else '#ED7D31' for k in vals]
            bars = ax.bar(vals.keys(), vals.values(), color=cols, alpha=.85, edgecolor='white')
            for bar, v in zip(bars, vals.values()):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + .01,
                        f'{v:.3f}', ha='center', va='bottom', fontsize=8)
            ax.set_ylim(0, 1); ax.set_ylabel(label); ax.set_title(f'Per-Client {label}')
            ax.grid(True, alpha=.3, axis='y')
            plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
        axes[0].legend(handles=[Patch(color='#4472C4', label='PhraseBank'),
                                 Patch(color='#ED7D31', label='Twitter')])
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'plots', 'per_client.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('No per-client metrics available.')
except Exception as e:
    print(f'Per-client plot error: {e}')


---
## Step 8 — Save & Download Results

In [ ]:
import json, datetime, os

summary = {
    'experiment': {
        'preset': PRESET, 'timestamp': datetime.datetime.now().isoformat(),
        'device': DEVICE, 'num_rounds': cfg.federated.num_rounds,
        'num_clients': cfg.federated.num_clients, 'lora_rank': cfg.lora.rank,
        'optimal_experts_M': int(pipeline.server.n_experts),
    },
    'final_metrics': {},
    'communication': results.get('comm_stats', {}),
}
for ds, m in results.get('final_metrics', {}).items():
    a = m.get('aggregate', {})
    summary['final_metrics'][ds] = {
        'mean_accuracy':    round(a.get('mean_accuracy',    0), 4),
        'mean_macro_f1':    round(a.get('mean_macro_f1',    0), 4),
        'mean_weighted_f1': round(a.get('mean_weighted_f1', 0), 4),
    }

rpath = os.path.join(OUTPUT_DIR, 'results', 'experiment_summary.json')
os.makedirs(os.path.dirname(rpath), exist_ok=True)
with open(rpath, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Saved: {rpath}')
print(json.dumps(summary, indent=2, default=str))


In [ ]:
import os

zip_path = '/content/fedlease_results.zip'
os.system(f'zip -r "{zip_path}" "{OUTPUT_DIR}" -x "*.pt"')
print(f'{os.path.getsize(zip_path)/1e6:.1f} MB')
try:
    from google.colab import files
    files.download(zip_path)
    print('Download started')
except ImportError:
    print(f'Results at: {OUTPUT_DIR}')


---
## Appendix — Inspect & Ablations

In [ ]:
try:
    model = pipeline.clients[0].model
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total         : {tot:>12,}')
    print(f'Trainable     : {trn:>12,}  ({trn/tot*100:.4f}%)')
    print(f'Frozen        : {tot-trn:>12,}')
    print(f'Upload/round  : {trn*4/1e6:.3f} MB (fp32)')
    print(f'Full-model    : {tot*4/1e6:.0f} MB  ({tot//trn}x larger)')
except Exception as e:
    print(f'Inspection failed: {e}')


In [ ]:
# IID control -- uncomment to run
# Expect: low silhouette, M* near 1, no improvement over FedAvg-LoRA
#
# from utils.config import merge_configs
# cfg_iid = merge_configs(cfg, {
#     'data':  {'partition_strategy': 'iid'},
#     'paths': {'output_dir': OUTPUT_DIR + '_iid'},
# })
# cdl_iid = FederatedDataPartitioner(seed=42).partition(
#     phraseb_splits, twitter_splits, num_clients=10, strategy='iid')
# pl_iid  = FedLEASEPipeline(config=cfg_iid, client_data_list=cdl_iid,
#                             test_loaders=test_loaders, device=device)
# res_iid = pl_iid.run()
# print('IID M*:', pl_iid.server.n_experts)
print('IID ablation -- uncomment to run')


---
## Troubleshooting

| Error | Fix |
|---|---|
| `Dataset scripts are no longer supported` | Re-run Step 1 (pin datasets<3.0) then restart runtime |
| `trust_remote_code is not supported` | Same — you have datasets>=3.0 |
| `HTTPError 404` from datasets-server | Re-run Step 3 (patch uses hf_hub_download now) |
| `ModuleNotFoundError` | Re-run Steps 1 and 2 |
| `CUDA out of memory` | Set `PRESET='debug'` or `batch_size=16` |
| `NameError: phraseb_splits` | Prior cell failed — fix error, rerun that cell first |
| Session disconnected | Re-run Steps 5 -> 6 (data re-downloads) |

In [ ]:
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total')
print('Set PRESET=\'debug\' in Step 5 and re-run.')
